In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA

In [3]:
df = pd.read_csv("01 covid_toy.csv")

In [4]:
display(df.head())

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
print(df.dtypes)

age            int64
gender        object
fever        float64
cough         object
city          object
has_covid     object
dtype: object


In [6]:
numeric_columns = df.select_dtypes(include=np.number).columns
print("Numric columns: ")
print(numeric_columns)

Numric columns: 
Index(['age', 'fever'], dtype='object')


In [7]:

categorical_columns = df.select_dtypes(include="object").columns
print("Categorical columns")
print(categorical_columns)

Categorical columns
Index(['gender', 'cough', 'city', 'has_covid'], dtype='object')


In [9]:
df["has_covid"] = df["has_covid"].astype("object")

In [10]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [11]:
df = df.replace("?",np.nan)

In [12]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [13]:
df["fever"]= pd.to_numeric(df["fever"])

In [14]:
df["fever"]=df["fever"].fillna(df["fever"].mean())

In [15]:
print(df.isnull().sum())

age          0
gender       0
fever        0
cough        0
city         0
has_covid    0
dtype: int64


In [16]:
scaler = MinMaxScaler().set_output(transform="pandas")
scaled_data = scaler.fit_transform(df[numeric_columns])
scaled_data.head()

,age,fever
0,0.696203,0.833333
1,0.278481,0.333333
2,0.468354,0.500000
3,0.329114,0.000000
4,0.759494,0.500000


In [17]:
categorical_columns = ["city","has_covid"]

In [18]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
).set_output(transform="pandas")

encoded_data = encoder.fit_transform(df[categorical_columns])
display(encoded_data.head())

,city_Bangalore,city_Delhi,city_Kolkata,city_Mumbai,has_covid_No,has_covid_Yes
0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,1.0,0.0
3,0.0,0.0,1.0,0.0,1.0,0.0
4,0.0,0.0,0.0,1.0,1.0,0.0


In [20]:
X = df.drop("has_covid", axis=1)
y = df["has_covid"]

In [22]:
X_numeric = X.select_dtypes(include=np.number)
y_numeric = y.map({'Yes': 1, 'No': 0})

selector = SelectKBest(
    score_func = f_regression,
    k=2
).set_output(transform="pandas")

x_selected = selector.fit_transform(X_numeric, y_numeric)
print(x_selected.head())

   age  fever
0   60  103.0
1   27  100.0
2   42  101.0
3   31   98.0
4   65  101.0
